Cata y Manu, algunos comentarios:
- El código está impecable (como siempre). Todo prolijo, comentado, detallado. Un lujo

# Big Data: Trabajo Práctico 2 - Grupo 5

In [ ]:
# importamos paquetes
import pandas as pd
import os
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.cluster import KMeans, AgglomerativeClustering
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import GridSearchCV
from sklearn.neighbors import KernelDensity
from scipy.cluster.hierarchy import dendrogram, linkage, cut_tree

In [ ]:
# ---------------------------------------------------------------
# Rutas del proyecto. El notebook vive en programs/ y lee de
# ../input/ y escribe en ../output/. Funciona tanto si se ejecuta
# desde programs/ como desde la raiz del TP.
# ---------------------------------------------------------------
from pathlib import Path

BASE = Path.cwd() if (Path.cwd() / "input").is_dir() else Path.cwd().parent
INPUT = BASE / "input"
OUTPUT = BASE / "output"
OUTPUT.mkdir(exist_ok=True)

print("Input :", INPUT)
print("Output:", OUTPUT)

## PARTE I: limpieza de la base de datos y la maldición de la multidimensionalidad

In [ ]:
# importamos y juntamos las bases de dato
df1 = pd.read_excel(INPUT / "usu_individual_T124.xlsx")
df2 = pd.read_stata(INPUT / "usu_individual_T104.dta", convert_categoricals=False)
etiquetas = pd.read_stata(INPUT / "usu_individual_T104.dta")

df1 = df1[df1["REGION"] == 42]
df2 = df2[df2["region"] == 42]
etiquetas = etiquetas[etiquetas["region"] == 'Cuyo']

df2.columns = df2.columns.str.upper()
etiquetas.columns = etiquetas.columns.str.upper()

formato_incorrecto = ['CH14', 'CH15_COD', 'CH16_COD', 'PP04B_COD', 'PP04D_COD', 'PP11B_COD', 'PP11D_COD', 'DECOCUR',  'IDECOCUR',  'RDECOCUR',  'GDECOCUR',  'PDECOCUR',  'ADECOCUR',  'DECINDR',  'IDECINDR',  'RDECINDR',  'GDECINDR',  'PDECINDR',  'ADECINDR',  'DECIFR',  'IDECIFR',  'RDECIFR',  'GDECIFR',  'PDECIFR',  'ADECIFR',  'DECCFR',  'IDECCFR',  'RDECCFR',  'GDECCFR',  'PDECCFR',  'ADECCFR']
for col in formato_incorrecto:
    df2[col] = pd.to_numeric(df2[col], errors='coerce')

df = pd.concat([df2, df1], ignore_index=True)

In [ ]:
# hacemos la misma limpieza del TP1
df['CH06'] = df['CH06'].astype(float).replace(-1.0, 0)
df['CH14'] = df['CH14'].astype(float).replace(98.0, float("NaN"))
df['PP06C'] = df['PP06C'].astype(float).replace(-8.0, float('NaN'))
df['PP06C'] = df['PP06C'].astype(float).replace(-7.0, float('NaN'))
df['PP06D'] = df['PP06D'].astype(float).replace(-8.0, float('NaN'))
df['PP06D'] = df['PP06D'].astype(float).replace(-7.0, float('NaN'))

columnas_unicas = set(df1.columns).symmetric_difference(set(df2.columns))
df = df.drop(columns=[col for col in columnas_unicas if col in df.columns])

df['MAS_500'] = df['MAS_500'].astype(str).replace('S', 1)
df['MAS_500'] = df['MAS_500'].astype(str).replace('N', 0)
df['MAS_500'] = pd.to_numeric(df['MAS_500'], errors='coerce')

df.loc[df['CH09'] == 3, 'CH14'] = df.loc[df['CH09'] == 3, 'CH14'].astype(float).replace(float("NaN"), 0.0)
df.loc[df['CH10'] == 3, 'CH14'] = df.loc[df['CH10'] == 3, 'CH14'].astype(float).replace(float("NaN"), 0.0)
df.loc[~df['CH15'].isin([3.0, 4.0, 5.0]), 'CH15_COD'] = 0
df.loc[~df['CH16'].isin([3.0, 4.0, 5.0]), 'CH16_COD'] = 0
df.loc[df['PP04A'] != 3.0, 'PP04B_COD'] = 0
df.loc[~df['PP06E'].isin([2.0, 3.0]), 'PP06H'] = 0

df['PP05F'] = df['PP05F'].astype(float).replace(6.0, 1)
df['PP05F'] = df['PP05F'].astype(float).replace(7.0, 2)

df['PP07F5'] = df['PP07F5'].astype(float).replace(5.0, 1)
df['PP07G_59'] = df['PP07G_59'].astype(float).replace(5.0, 1)

columnas_ocupados = ['PP03C', 'PP03D', 'PP3E_TOT', 'PP3F_TOT', 'PP03G', 'PP03H', 'PP03I', 'PP03J', 'INTENSI', 'PP04A', 'PP04B_COD', 'PP04B1', 'PP04B2', 'PP04B3_MES', 'PP04B3_ANO', 'PP04B3_DIA', 'PP04C', 'PP04C99', 'PP04D_COD', 'PP04G', 'PP05B2_MES', 'PP05B2_ANO', 'PP05B2_DIA', 'PP05C_1', 'PP05C_2', 'PP05C_3', 'PP05E', 'PP05F', 'PP05H', 'PP06A', 'PP06E', 'PP06H', 'PP07A', 'PP07C', 'PP07D', 'PP07E', 'PP07F1', 'PP07F2', 'PP07F3', 'PP07F4', 'PP07F5', 'PP07G1', 'PP07G2', 'PP07G3', 'PP07G4', 'PP07G_59', 'PP07H', 'PP07I', 'PP07J', 'PP07K']
for col in columnas_ocupados:
    df.loc[df['ESTADO'] != 1, col] = df.loc[df['ESTADO'] != 1, col].astype(float).replace(float("NaN"), 0.0)

columnas_desocupados = ['PP10A', 'PP10C', 'PP10D', 'PP10E', 'PP11A', 'PP11B_COD', 'PP11B1', 'PP11B2_MES', 'PP11B2_ANO', 'PP11B2_DIA', 'PP11C', 'PP11C99', 'PP11D_COD', 'PP11G_ANO', 'PP11G_MES', 'PP11G_DIA', 'PP11L', 'PP11L1', 'PP11N', 'PP11O', 'PP11P', 'PP11Q', 'PP11R', 'PP11S', 'PP11T']
for col in columnas_desocupados:
    df.loc[df['ESTADO'] != 2, col] = df.loc[df['ESTADO'] != 2, col].astype(float).replace(float("NaN"), 0.0)

eliminar = ['IMPUTA', 'PP09A', 'PP09A_ESP', 'PP09B', 'PP09C', 'PP09C_ESP'] 
df = df.drop(columns=[col for col in eliminar if col in df.columns])

columnas_float = ['PP06C', 'PP06D', 'PP08D1', 'PP08D4', 'PP08F1', 'PP08F2', 'PP08J1', 'PP08J2', 'PP08J3', 'P21', 'TOT_P12', 'P47T', 'V2_M', 'V3_M', 'V4_M', 'V5_M', 'V8_M', 'V9_M', 'V10_M', 'V11_M', 'V12_M', 'V18_M', 'V19_AM', 'V21_M', 'T_VI', 'ITF', 'IPCF']
for col in df.columns:
    if col not in columnas_float and col != "CODUSU":
        df[col] = df[col].astype('Int64')

for col in columnas_float:
    df[col] = df[col].astype('Float64')

In [ ]:
# volvemos a crear la base de datos "respondieron" y le agregamos columnas relevantes para el análisis
respondieron = pd.DataFrame(df[df['ESTADO'] != 0])
respondieron['DESOCUPADOS'] = (df['ESTADO'] == 2).astype(int)

### Ejercicio 1: tamaño base de datos

In [ ]:
# tamaño de la base de datos
print(f"La base de datos tiene {respondieron.shape[1]} columnas.")

### Ejercicio 2: tamaño base de datos con dummy

In [ ]:
columnas = 0
omitir = ['NRO_HOGAR', 'PP08F2', 'PP08J2', 'PP08J3', 'PP04B3_MES', 'PP04B3_ANO', 'PP04B3_DIA', 'PP05B2_MES', 'PP05B2_ANO', 'PP05B2_DIA', 'PP11B2_MES', 'PP11B2_ANO', 'PP11B2_DIA', 'PP11C', 'PP11G_ANO', 'PP11G_MES', 'PP11G_DIA','DECOCUR', 'IDECOCUR', 'RDECOCUR', 'GDECOCUR', 'PDECOCUR', 'ADECOCUR', 'DECINDR', 'IDECINDR', 'RDECINDR', 'GDECINDR', 'PDECINDR', 'ADECINDR', 'V3_M', 'V4_M', 'V9_M', 'DECIFR', 'IDECIFR', 'RDECIFR', 'GDECIFR', 'PDECIFR', 'ADECIFR', 'DECCFR', 'IDECCFR', 'RDECCFR', 'GDECCFR', 'PDECCFR', 'ADECCFR']

for col in respondieron.columns:
    if col in omitir:
        continue
    elementos = set(respondieron[col])
    elementos.discard(pd.NA)
    if 0 in elementos:
        elementos.remove(0)
    if len(elementos) < 9 & 9 in elementos:
        elementos.remove(9)
    if len(elementos) < 15 & 99 in elementos:
        elementos.remove(99)
    if len(elementos) <= 2:
        columnas += 1
    elif len(elementos) > 15:
        columnas += 1
    else:
        columnas += len(elementos)

print(f"La base de datos tendría {columnas} columnas.")

### Ejercicio 3: creación dummies

In [ ]:
# creamos función para tomar los valores unicos de la columna manteniendo el orden en el que aparecen
def valores_unicos(base, columna):
    valores = []
    for x in list(base[columna]):
        if x not in valores:
            valores.append(x)
    return valores

In [ ]:
# creamos las dummies necesarias
dummies = ['AGLOMERADO', 'CH04', 'CH07', 'CH08', 'CH15', 'CH16', 'NIVEL_ED']
labels = []
valores = []

for var in dummies:
    labels.append(valores_unicos(etiquetas, var))
    valores.append(valores_unicos(respondieron, var))
for label, valor in zip(labels, valores):
    for nombre, numero in zip(label, valor):
        respondieron[nombre.upper()] = (respondieron[var] == numero).astype(int)

## PARTE II: clustering

### Ejercicio 1: agrupación desocupados

In [ ]:
# lista de variables relevantes elegidas
variables_relevantes = ['MAS_500', 'CH06', 'PP02I', 'PP10D', 'IPCF', 'GRAN MENDOZA', 'SAN LUIS - EL CHORRILLO', 'GRAN SAN JUAN', 
                        'VARÓN', 'MUJER', 'SEPARADO O DIVORCIADO', 'SOLTERO', 'VIUDO', 'CASADO', 'UNIDO', 'OBRA SOCIAL (INCLUYE PAMI)', 
                        'NO PAGA NI LE DESCUENTAN', 'MUTUAL/PREPAGA/SERVICIO DE EMERGENCIA', 
                        'OBRA SOCIAL Y MUTUAL/PREPAGA/SERVICIO DE EMERGENCIA', 'PLANES Y SEGUROS PÚBLICOS', 'EN ESTA LOCALIDAD', 
                        'EN OTRA PROVINCIA (ESPECIFICAR)', 'EN OTRA LOCALIDAD', 'EN UN PAÍS LIMÍTROFE', 'EN OTRO PAÍS', 
                        'EN ESTA LOCALIDAD', 'NO HABÍA NACIDO', 'EN OTRA PROVINCIA (ESPECIFICAR)', 'EN OTRA LOCALIDAD DE ESTA PROVINCIA',
                        'EN UN PAÍS LIMÍTROFE', 'EN OTRO PAÍS', 'SIN INSTRUCCIÓN', 'PRIMARIA INCOMPLETA (INCLUYE EDUCACIÓN ESPECIAL)', 
                        'SUPERIOR UNIVERSITARIA INCOMPLETA', 'SUPERIOR UNIVERSITARIA COMPLETA', 'SECUNDARIA INCOMPLETA', 
                        'PRIMARIA COMPLETA', 'SECUNDARIA COMPLETA']

### Ejercicio 2: K-Medias

#### a) Algoritmo para k=2, k=4 y k=10

In [ ]:
# definimos función educación para calcular los años de educación
primaria = 6
egb = 9
secundaria = 12
terciario = 14
universidad = 16
posgrado = 18

def calcular_educacion(fila):
    nivel = fila['CH12']
    finalizado = ['CH13']
    aprobado = fila['CH14']

    if nivel == 1:
        return 0
    elif nivel == 2:
        if finalizado == 1:
            return primaria
        else: 
            if pd.isna(aprobado) or aprobado == 99:
                return np.nan
            return aprobado
    elif nivel == 3:
        if finalizado == 1:
            return egb
        else: 
            if pd.isna(aprobado) or aprobado == 99:
                return np.nan
            return aprobado
    elif nivel == 4:
        if finalizado == 1:
            return secundaria
        else: 
            if pd.isna(aprobado) or aprobado == 99:
                return np.nan
            return primaria + aprobado
    elif nivel == 5:
        if finalizado == 1:
            return secundaria
        else: 
            if pd.isna(aprobado) or aprobado == 99:
                return np.nan
            return egb + aprobado
    elif nivel == 6:
        if finalizado == 1:
            return terciario
        else: 
            if pd.isna(aprobado) or aprobado == 99:
                return np.nan
            return secundaria + aprobado
    elif nivel == 7:
        if finalizado == 1:
            return universidad
        else: 
            if pd.isna(aprobado) or aprobado == 99:
                return np.nan
            return secundaria + aprobado
    elif nivel == 8:
        if finalizado == 1:
            return posgrado
        else: 
            if pd.isna(aprobado) or aprobado == 99:
                return np.nan
            return universidad + aprobado
    else:
        return 0

In [ ]:
# creamos variable EDUCACION (años de educación)
respondieron['EDUCACION'] = respondieron.apply(calcular_educacion, axis=1)

In [ ]:
# creamos variable INGRESO (con año base los precios de 2024)
coef_actualizacion = 627.2574
respondieron['INGRESO'] = respondieron.apply(lambda row: row['P47T'] * coef_actualizacion if row['ANO4'] == 2004 else row['P47T'], axis=1)

In [ ]:
# ajustamos los tres modelos
X = respondieron[['INGRESO', 'EDUCACION']].dropna().to_numpy()
kmeans2 = KMeans(n_clusters=2, random_state=10, init="random", n_init=20).fit(X)
kmeans4 = KMeans(n_clusters=4, random_state=10, init="random", n_init=20).fit(X)
kmeans10 = KMeans(n_clusters=10, random_state=10, init="random", n_init=20).fit(X)

In [ ]:
# graficamos 
fig, axs = plt.subplots(1, 3, figsize=(21, 7))

# gráfico k=2
axs[0].scatter(X[:, 0], X[:, 1], c=kmeans2.labels_)
axs[0].set_title("K-Means Clustering con K=2")
axs[0].set_xlabel("Ingreso (base 2024)")
axs[0].set_ylabel("Años de educación")

# gráfico k=4
axs[1].scatter(X[:, 0], X[:, 1], c=kmeans4.labels_)
axs[1].set_title("K-Means Clustering con K=4")
axs[1].set_xlabel("Ingreso (base 2024)")

# gráfico k=10
axs[2].scatter(X[:, 0], X[:, 1], c=kmeans10.labels_)
axs[2].set_title("K-Means Clustering con K=10")
axs[2].set_xlabel("Ingreso (base 2024)")

plt.tight_layout()
plt.savefig(OUTPUT / "k-medias.png", dpi=300, bbox_inches='tight')
plt.show()

#### b) Agregamos condición ocupación 

In [ ]:
# acomodamos los datos para graficar y ajustamos modelos
filtrado = respondieron[['INGRESO', 'EDUCACION', 'ESTADO']].dropna()
X_ocupados = filtrado[filtrado['ESTADO'] == 1][['INGRESO', 'EDUCACION']].to_numpy()
X_desocupados = filtrado[filtrado['ESTADO'] == 2][['INGRESO', 'EDUCACION']].to_numpy()
kmeans_ocupados = KMeans(n_clusters=2, random_state=10, init="random", n_init=20).fit(X_ocupados)
kmeans_desocupados = KMeans(n_clusters=2, random_state=10, init="random", n_init=20).fit(X_desocupados)

In [ ]:
# graficamos
fig, axs = plt.subplots(1, 2, figsize=(14, 7))

# gráfico ocupados
axs[0].scatter(X_ocupados[:, 0], X_ocupados[:, 1], c=kmeans_ocupados.labels_)
axs[0].set_title("K-Means Clustering con K=2 - Ocupados")
axs[0].set_xlabel("Ingreso (base 2024)")
axs[0].set_ylabel("Años de educación")

# gráfico desocupados
axs[1].scatter(X_desocupados[:, 0], X_desocupados[:, 1], c=kmeans_desocupados.labels_)
axs[1].set_title("K-Means Clustering con K=2 - Desocupados")
axs[1].set_xlabel("Ingreso (base 2024)")

plt.tight_layout()
plt.savefig(OUTPUT / "kmedias_comparacion.png", dpi=300, bbox_inches='tight')
plt.show()

### Ejercicio 3: clustering jerárquico

In [ ]:
# acomodamos datos y hacemos análisis de clustering jerárquico
scaler = StandardScaler()
datos = scaler.fit_transform(respondieron[variables_relevantes])
linkage_datos = linkage(datos, metric='euclidean')

In [ ]:
# graficamos con un dendograma
fig, ax = plt.subplots(figsize=(15,10))
dendrogram(linkage_datos, truncate_mode="level", p=5)
plt.savefig(OUTPUT / "dendograma.png", dpi=300, bbox_inches='tight')

## PARTE III: histogramas y kernels

### Ejercicio 1: distribución ingreso individual

In [ ]:
# acomodamos datos para graficar (tratamiento de outliers)
filtrado = respondieron.copy()
filtrado = filtrado[filtrado['P47T'].notna() & (filtrado['P47T'] > 0)]

filtrado_2004 = filtrado[filtrado['ANO4'] == 2004]
p75_2004 = filtrado_2004['P47T'].quantile(0.75)
filtrado_2004 = filtrado_2004[filtrado_2004['P47T'] <= p75_2004]

filtrado_2024 = filtrado[filtrado['ANO4'] == 2024]
p75_2024 = filtrado_2024['P47T'].quantile(0.75)
filtrado_2024 = filtrado_2024[filtrado_2024['P47T'] <= p75_2024]

In [ ]:
# graficamos 2004
fig, axes = plt.subplots(1, 2, figsize=(14, 6), sharey=True)

# gráfico 2004
sns.histplot(data=filtrado_2004, x='P47T', bins=30, kde=True,
             color='steelblue', ax=axes[0])
axes[0].set_title('Distribución de Ingreso Individual (2004)')
axes[0].set_xlabel("Ingreso")
axes[0].set_ylabel("Frecuencia")

# gráfico 2024
sns.histplot(data=filtrado_2024, x='P47T', bins=30, kde=True,
             color='steelblue', ax=axes[1])
axes[1].set_title('Distribución de Ingreso Individual (2024)')
axes[1].set_xlabel("Ingreso")
axes[1].set_ylabel("Frecuencia")

plt.tight_layout()
plt.savefig(OUTPUT / "histograma.png", dpi=300)
plt.show()

### Ejercicio 2: una observación por hogar

#### a) y b) generamos variable desocupados y variable ingreso por hogar

In [ ]:
# creamos base de datos agrupada por hogares, con la columna desocupado (toma valor 1 si hay un desocupado en el hogar)
hogares = respondieron.groupby('CODUSU', as_index=False).agg({'ANO4': 'min', 'DESOCUPADOS': 'max', 'P47T': lambda x: x[x > 0].sum()})
hogares.columns = ['id_hogar', 'anio', 'hay_desocupado', 'ingreso_hogar']

#### c) distribución del ingreso por hogar

In [ ]:
# acomodamos datos para graficar
filtrado = hogares.copy()
filtrado = filtrado[filtrado['ingreso_hogar'].notna() & (filtrado['ingreso_hogar'] > 0)]

filtrado_2004 = filtrado[filtrado['anio'] == 2004]
p75_2004 = filtrado_2004['ingreso_hogar'].quantile(0.75)
filtrado_2004 = filtrado_2004[filtrado_2004['ingreso_hogar'] <= p75_2004]
data_2004_desocup = filtrado_2004[filtrado_2004['hay_desocupado'] == 1]['ingreso_hogar'].values.reshape(-1, 1)
data_2004_ocup = filtrado_2004[filtrado_2004['hay_desocupado'] == 0]['ingreso_hogar'].values.reshape(-1, 1)

filtrado_2024 = filtrado[filtrado['anio'] == 2024]
p75_2024 = filtrado_2024['ingreso_hogar'].quantile(0.75)
filtrado_2024 = filtrado_2024[filtrado_2024['ingreso_hogar'] <= p75_2024]
data_2024_desocup = filtrado_2024[filtrado_2024['hay_desocupado'] == 1]['ingreso_hogar'].values.reshape(-1, 1)
data_2024_ocup = filtrado_2024[filtrado_2024['hay_desocupado'] == 0]['ingreso_hogar'].values.reshape(-1, 1)

In [ ]:
# graficamos 2004
kde_2004_d = KernelDensity(kernel='tophat', bandwidth=100).fit(data_2004_desocup)
kde_2004_o = KernelDensity(kernel='tophat', bandwidth=100).fit(data_2004_ocup)

x_2004 = np.linspace(filtrado_2004['ingreso_hogar'].min(), filtrado_2004['ingreso_hogar'].max(), 1000).reshape(-1, 1)

log_dens_2004_d = kde_2004_d.score_samples(x_2004)
log_dens_2004_o = kde_2004_o.score_samples(x_2004)

fig, ax = plt.subplots(1, 1)
fig.set_figheight(6)
fig.set_figwidth(8)
fig.suptitle("Distribución de Ingreso del Hogar (2004) - Kernel Uniforme")
ax.plot(x_2004[:, 0], np.exp(log_dens_2004_d), color='red', label='Con desocupados')
ax.plot(x_2004[:, 0], np.exp(log_dens_2004_o), color='green', label='Sin desocupados')
ax.set_xlabel("Ingreso Hogar")
ax.set_ylabel("Densidad")
ax.legend()
ax.grid(True)

plt.tight_layout()
plt.savefig(OUTPUT / "distribucion2004.png", dpi=300)
plt.show()

In [ ]:
# graficamos 2024
kde_2024_d = KernelDensity(kernel='tophat', bandwidth=100000).fit(data_2024_desocup)
kde_2024_o = KernelDensity(kernel='tophat', bandwidth=100000).fit(data_2024_ocup)

x_2024 = np.linspace(filtrado_2024['ingreso_hogar'].min(), filtrado_2024['ingreso_hogar'].max(), 1000).reshape(-1, 1)

log_dens_2024_d = kde_2024_d.score_samples(x_2024)
log_dens_2024_o = kde_2024_o.score_samples(x_2024)

fig, ax = plt.subplots(1, 1)
fig.set_figheight(6)
fig.set_figwidth(8)
fig.suptitle("Distribución de Ingreso del Hogar (2024) - Kernel Uniforme")
ax.plot(x_2024[:, 0], np.exp(log_dens_2024_d), color='red', label='Con desocupados')
ax.plot(x_2024[:, 0], np.exp(log_dens_2024_o), color='green', label='Sin desocupados')
ax.set_xlabel("Ingreso Hogar")
ax.set_ylabel("Densidad")
ax.legend()
ax.grid(True)
plt.tight_layout()
plt.savefig(OUTPUT / "distribucion2024.png", dpi=300)
plt.show()

#### e) diferencia entre hogares con y sin desocupado

In [ ]:
# calculamos diferencias
ingreso_desocupados_2004 = hogares[(hogares['anio'] == 2004) & (hogares['hay_desocupado'] == 1)]['ingreso_hogar'].mean()
ingreso_ocupados_2004 = hogares[(hogares['anio'] == 2004) & (hogares['hay_desocupado'] == 0)]['ingreso_hogar'].mean()
diferencia_2004 = (ingreso_ocupados_2004 - ingreso_desocupados_2004)/ingreso_desocupados_2004

ingreso_desocupados_2024 = hogares[(hogares['anio'] == 2024) & (hogares['hay_desocupado'] == 1)]['ingreso_hogar'].mean()
ingreso_ocupados_2024 = hogares[(hogares['anio'] == 2024) & (hogares['hay_desocupado'] == 0)]['ingreso_hogar'].mean()
diferencia_2024 = (ingreso_ocupados_2024 - ingreso_desocupados_2024)/ingreso_desocupados_2024

print(f"En 2004, la diferencia porcentual en ingreso fue {round(diferencia_2004 * 100)}%.")
print(f"En 2024, la diferencia porcentual en ingreso fue {round(diferencia_2024 * 100)}%.")

### Ejercicio 3: kernel gaussiano y el kernel de Epanechnikov

In [ ]:
# kernel gaussiano 2004
kde_2004_d = KernelDensity(kernel='gaussian', bandwidth=100).fit(data_2004_desocup)
kde_2004_o = KernelDensity(kernel='gaussian', bandwidth=100).fit(data_2004_ocup)

x_2004 = np.linspace(filtrado_2004['ingreso_hogar'].min(), filtrado_2004['ingreso_hogar'].max(), 1000).reshape(-1, 1)

log_dens_2004_d = kde_2004_d.score_samples(x_2004)
log_dens_2004_o = kde_2004_o.score_samples(x_2004)

fig, ax = plt.subplots(1, 1)
fig.set_figheight(6)
fig.set_figwidth(8)
fig.suptitle("Distribución de Ingreso del Hogar (2004) - Kernel Gaussiano")
ax.plot(x_2004[:, 0], np.exp(log_dens_2004_d), color='red', label='Con desocupados')
ax.plot(x_2004[:, 0], np.exp(log_dens_2004_o), color='green', label='Sin desocupados')
ax.set_xlabel("Ingreso Hogar")
ax.set_ylabel("Densidad")
ax.legend()
ax.grid(True)

plt.tight_layout()
plt.savefig(OUTPUT / "distribuciong2004.png", dpi=300)
plt.show()

In [ ]:
# kernel gaussiano 2024
kde_2024_d = KernelDensity(kernel='gaussian', bandwidth=100000).fit(data_2024_desocup)
kde_2024_o = KernelDensity(kernel='gaussian', bandwidth=100000).fit(data_2024_ocup)

x_2024 = np.linspace(filtrado_2024['ingreso_hogar'].min(), filtrado_2024['ingreso_hogar'].max(), 1000).reshape(-1, 1)

log_dens_2024_d = kde_2024_d.score_samples(x_2024)
log_dens_2024_o = kde_2024_o.score_samples(x_2024)

fig, ax = plt.subplots(1, 1)
fig.set_figheight(6)
fig.set_figwidth(8)
fig.suptitle("Distribución de Ingreso del Hogar (2024) - Kernel Gaussiano")
ax.plot(x_2024[:, 0], np.exp(log_dens_2024_d), color='red', label='Con desocupados')
ax.plot(x_2024[:, 0], np.exp(log_dens_2024_o), color='green', label='Sin desocupados')
ax.set_xlabel("Ingreso Hogar")
ax.set_ylabel("Densidad")
ax.legend()
ax.grid(True)
plt.tight_layout()
plt.savefig(OUTPUT / "distribuciong2024.png", dpi=300)
plt.show()

In [ ]:
# kernel de Epanechnikov 2004
kde_2004_d = KernelDensity(kernel='epanechnikov', bandwidth=100).fit(data_2004_desocup)
kde_2004_o = KernelDensity(kernel='epanechnikov', bandwidth=100).fit(data_2004_ocup)

x_2004 = np.linspace(filtrado_2004['ingreso_hogar'].min(), filtrado_2004['ingreso_hogar'].max(), 1000).reshape(-1, 1)

log_dens_2004_d = kde_2004_d.score_samples(x_2004)
log_dens_2004_o = kde_2004_o.score_samples(x_2004)

fig, ax = plt.subplots(1, 1)
fig.set_figheight(6)
fig.set_figwidth(8)
fig.suptitle("Distribución de Ingreso del Hogar (2004) - Kernel Epanechnikov")
ax.plot(x_2004[:, 0], np.exp(log_dens_2004_d), color='red', label='Con desocupados')
ax.plot(x_2004[:, 0], np.exp(log_dens_2004_o), color='green', label='Sin desocupados')
ax.set_xlabel("Ingreso Hogar")
ax.set_ylabel("Densidad")
ax.legend()
ax.grid(True)

plt.tight_layout()
plt.savefig(OUTPUT / "distribucione2004.png", dpi=300)
plt.show()

In [ ]:
# kernel de Epanechnikov 2024
kde_2024_d = KernelDensity(kernel='epanechnikov', bandwidth=100000).fit(data_2024_desocup)
kde_2024_o = KernelDensity(kernel='epanechnikov', bandwidth=100000).fit(data_2024_ocup)

x_2024 = np.linspace(filtrado_2024['ingreso_hogar'].min(), filtrado_2024['ingreso_hogar'].max(), 1000).reshape(-1, 1)

log_dens_2024_d = kde_2024_d.score_samples(x_2024)
log_dens_2024_o = kde_2024_o.score_samples(x_2024)

fig, ax = plt.subplots(1, 1)
fig.set_figheight(6)
fig.set_figwidth(8)
fig.suptitle("Distribución de Ingreso del Hogar (2024) - Kernel Epanechnikov")
ax.plot(x_2024[:, 0], np.exp(log_dens_2024_d), color='red', label='Con desocupados')
ax.plot(x_2024[:, 0], np.exp(log_dens_2024_o), color='green', label='Sin desocupados')
ax.set_xlabel("Ingreso Hogar")
ax.set_ylabel("Densidad")
ax.legend()
ax.grid(True)
plt.tight_layout()
plt.savefig(OUTPUT / "distribucione2024.png", dpi=300)
plt.show()

FIN :)